# PARTIE III : GÉNÉRATION DE DONNÉES SYNTHÉTIQUES



## INTRODUCTION ET OBJECTIF : Trouver le meilleur compromis entre l'utilité des données et la confidentialité
Ce rapport présente les résultats de la génération de données synthétiques à partir d'un jeu de données réel (trainDataDC2.csv.gz) en utilisant une approche basée sur la méthode des "Avatars" (Guillaudeux et al., 2023)

L'objectif principal de cette démarche est de trouver un compromis optimal entre l'utilité des données (réalisme statistique et conservation des corrélations) et la confidentialité (anonymat des patients).

Les données synthétiques sont cruciales dans de nombreux domaines, notamment la recherche médicale et l'analyse de données, où l'accès direct aux données sensibles est souvent restreint en raison de préoccupations de confidentialité.

### Méthode (Avatar - Guillaudeux et al. 2023)

La méthode "Avatar" vise à créer des jeux de données artificiels qui conservent les propriétés statistiques des données originales tout en garantissant que les individus réels ne peuvent pas être ré-identifiés. Elle repose sur l'identification des 'k' plus proches voisins (k-NN) pour chaque patient réel. Un "Avatar" (un patient virtuel) est ensuite créé en calculant un point aléatoire situé au milieu de ces voisins (barycentre pondéré). Cette approche locale garantit à la fois le réalisme (utilité) et l'anonymat (confidentialité).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import cdist
import warnings

#Désactivation des alertes non critiques pour garder une console propre
warnings.filterwarnings('ignore')


# 1. PARAMÉTRAGE ET PRÉPARATION DES DONNÉES

Le paramètre `K_VOISINS` (k) est crucial et contrôle le compromis entre l'Utilité et la Confidentialité des données synthétiques :
- **Si k est trop petit** (ex: 2) : l'avatar généré est très proche du patient d'origine. Les métriques d'utilité seront bonnes, mais la confidentialité sera mauvaise, augmentant le risque de ré-identification.
- **Si k est trop grand** (ex: 15) : l'avatar est une moyenne de patients très différents. La confidentialité sera bonne, car l'avatar sera très éloigné des individus réels, mais les corrélations originales pourraient être détruites, réduisant l'utilité des données.

### Préparation des Données

Cette étape inclut le chargement du jeu de données réel, l'identification des variables binaires, la gestion des valeurs manquantes et la standardisation des données:

1.  **Chargement des Données**:
    *   Le jeu de données réel (`trainDataDC2.csv.gz`) est chargé. La colonne `patid` (identifiant patient) est retirée car elle n'apporte aucune information médicale et ne doit pas influencer la génération des avatars.

2.  **Identification des Variables Binaires**:
    *   Les variables binaires sont identifiées pour un post-traitement spécifique (arrondi à 0 ou 1).

3.  **Gestion des Valeurs Manquantes (Imputation)**:
    *   Les valeurs manquantes sont imputées en utilisant la médiane de chaque colonne. Cette stratégie est préférée à la moyenne pour sa robustesse aux valeurs aberrantes (par exemple, des frais médicaux extrêmement élevés).

In [ ]:
K_VOISINS = 5
print("1. Chargement et préparation des données...")

#Chargement du jeu de données réel
#L'argument na_values='NA' assure que les textes 'NA' sont bien reconnus comme manquants
df_real = pd.read_csv('trainDataDC2.csv.gz', na_values='NA')

#L'identifiant patient ne contient aucune information médicale et ne doit pas influencer le calcul des distances. Nous le mettons de côté.
df_features = df_real.drop(columns=['patid'])
colonnes = df_features.columns

#Identification automatique des variables binaires (0 ou 1) nécessaire lors du post-traitement pour arrondir correctement les prédictions
cols_binaires = [c for c in colonnes if df_features[c].nunique() <= 2]

#Gestion des valeurs manquantes (Imputation)
#On utilise la médiane plutôt que la moyenne pour éviter que les valeurs aberrantes (frais médicaux élevés) ne faussent les remplacements.
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df_features)



1. Chargement et préparation des données...


### Standardisation des Données

L'algorithme k-NN calcule des distances euclidiennes. Si les variables ne sont pas mises à la même échelle (standardisées), une variable exprimée en milliers (comme les coûts médicaux) dominera le calcul des distances par rapport à une variable exprimée en dizaines (comme l'âge). La standardisation (transformant les données pour avoir une Moyenne=0 et un Écart-type=1) est appliquée pour équilibrer le poids de chaque variable dans l'espace de distance.

In [ ]:
#Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# 2. GÉNÉRATION DES AVATARS (k-NN Barycentre Aléatoire)

Cette étape fondamentale de la méthode "Avatar" consiste à créer des patients virtuels en se basant sur les *k* plus proches voisins des patients réels :

1.  **Ajustement du modèle k-NN**:
    *   L'algorithme des *k* plus proches voisins (k-NN) est ajusté sur les données standardisées. Le paramètre `K_VOISINS` a été fixé à 5 pour cette exécution. Ce paramètre est essentiel pour moduler le compromis utilité/confidentialité.

2.  **Création du Barycentre Aléatoire**:
    *   Pour chaque patient réel, l'algorithme identifie ses `k` plus proches voisins dans l'espace standardisé.
    *   Un "Avatar" est ensuite créé en calculant un barycentre pondéré aléatoire de ces `k` voisins. Les poids sont générés à partir d'une distribution de Dirichlet, garantissant que la somme des poids est égale à 1. Cela signifie que l'avatar est une combinaison linéaire des voisins, ce qui préserve les relations locales et ajoute un niveau d'aléatoire pour la confidentialité.

In [ ]:
print(f"2. Création des Avatars (k-NN avec k={K_VOISINS})")

#Ajustement du modèle des k plus proches voisins sur l'espace standardisé
knn = NearestNeighbors(n_neighbors=K_VOISINS)
knn.fit(X_scaled)

#Récupération des indices des k voisins pour chaque patient
distances, indices = knn.kneighbors(X_scaled)

#Initialisation de la matrice qui contiendra les données synthétiques
X_synth_scaled = np.zeros_like(X_scaled)

#Boucle de création : 1 patient réel = 1 avatar généré
for i in range(len(X_scaled)):
    #Extraction des coordonnées des k voisins du patient i
    voisins = X_scaled[indices[i]]

    # Création du barycentre aléatoire
    #La distribution de Dirichlet génère un vecteur de poids dont la somme vaut 1.
    #Exemple pour k=3 : [0.15, 0.55, 0.30].
    #L'avatar sera composé à 15% du voisin 1, 55% du voisin 2, etc.
    poids = np.random.dirichlet(np.ones(K_VOISINS))

    #Produit matriciel pour obtenir les coordonnées exactes de l'avatar
    X_synth_scaled[i] = np.dot(poids, voisins)

2. Création des Avatars (k-NN avec k=5)


# 3. POST-TRAITEMENT DES DONNÉES SYNTHÉTIQUES

Après la génération des avatars dans l'espace standardisé, un post-traitement est nécessaire pour rendre les données synthétiques cohérentes et exploitables :

1.  **Inversion de la Standardisation**:
    *   Les données synthétiques sont inversées pour retrouver leur échelle et leurs unités d'origine (par exemple, l'âge en années, les coûts en dollars).

2.  **Application des Contraintes Métier**:
    *   Des contraintes sont appliquées pour assurer la cohérence et le réalisme des données, incluant :
        *   **Contrainte de positivité/minimum** : Les valeurs sont bornées par le minimum observé dans les données réelles (ex: un âge ou un coût ne peut pas être inférieur au minimum réel).
        *   **Contrainte sur les variables binaires** : Les valeurs générées pour ces variables sont arrondies à 0 ou 1 et clippées dans cet intervalle pour refléter leur nature binaire.
        *   **Contrainte sur les variables de comptage** : Les variables telles que le nombre de jours ou l'âge sont arrondies à l'entier le plus proche.

3.  **Ajout d'Identifiants Factices**:
    *   Des identifiants factices (`patid`) sont ajoutés aux patients synthétiques, commençant à un chiffre élevé pour éviter toute confusion avec les identifiants réels.

In [ ]:
print("3. Retraitement des variables (arrondis, bornes)")

#Inversion de la standardisation pour retrouver les unités d'origine (années, dollars...)
X_synth = scaler.inverse_transform(X_synth_scaled)
df_synth = pd.DataFrame(X_synth, columns=colonnes)

#Restauration de la cohérence métier des variables
for col in colonnes:
    #Contrainte de positivité ou de minimum
    #Un âge ou un coût médical ne peut pas être inférieur au minimum observé dans la réalité.
    min_reel = df_features[col].min()
    df_synth[col] = np.maximum(df_synth[col], min_reel)

    #Contrainte sur les variables indicatrices (binaires)
    #Les probabilités générées sont arrondies à 0 ou 1, et bloquées dans cet intervalle.
    if col in cols_binaires:
        df_synth[col] = np.round(df_synth[col]).clip(0, 1)

#Contrainte sur les variables de comptage (jours, nombre d'exacerbations, âge)
cols_entieres = ["pre_asthma_days", "post_index_exacerbations365", "total_pre_index_cannisters_365", "index_age"]
for col in cols_entieres:
    if col in df_synth.columns:
        df_synth[col] = np.round(df_synth[col])

#Création d'identifiants factices pour les patients synthétiques
#On démarre à un chiffre élevé pour ne pas les confondre avec les identifiants réels.
df_synth.insert(0, 'patid', range(1000000, 1000000 + len(df_synth)))

3. Retraitement des variables (arrondis, bornes)


# 4. ÉVALUATION DES PERFORMANCES

L'évaluation des performances des données synthétiques est cruciale pour quantifier le compromis entre leur utilité (fidélité aux données réelles) et leur confidentialité (protection de la vie privée). Trois métriques principales sont utilisées :

### A. Utilité Univariée : Distance de Wasserstein

La distance de Wasserstein (également connue sous le nom de distance du transport optimal ou distance du terre-plein) mesure le "coût" minimal de transformation d'une distribution en une autre. Une distance de 0 indique une correspondance parfaite entre les distributions. Pour rendre les distances comparables entre variables, chaque variable est divisée par son écart-type. Une valeur basse est souhaitable, indiquant une bonne préservation des distributions univariées.

### B. Utilité Multivariée : Différence des Corrélations

Cette métrique évalue la capacité de la méthode à préserver les relations complexes entre les variables. Elle est calculée comme la différence moyenne absolue entre les matrices de corrélation des données réelles et synthétiques. Une faible différence indique que les structures de corrélation ont été bien conservées, ce qui est crucial pour le réalisme multivarié des données synthétiques.

### C. Confidentialité : Distance au Plus Proche Voisin (DCR - Distance to Closest Record)

La DCR mesure la distance moyenne entre chaque avatar synthétique et son plus proche patient réel. Un sous-échantillon de 500 patients est utilisé pour le calcul afin d'optimiser le temps d'exécution. Une distance élevée est souhaitable pour garantir que les avatars sont suffisamment éloignés des patients réels, réduisant ainsi le risque de ré-identification et protégeant la vie privée des individus.

In [ ]:
print("ÉVALUATION DES PERFORMANCES")

#A. Utilité univariée
w_dists = []
for c in colonnes:
    std = df_features[c].std()
    if std > 0:
        # Division par l'écart-type pour rendre les distances comparables entre variables
        w = wasserstein_distance(df_features[c].dropna() / std, df_synth[c] / std)
        w_dists.append(w)
print(f"Distance de Wasserstein moyenne (Utilité)       : {np.mean(w_dists):.4f} (Viser le plus bas)")

#B. Utilité multivariée
corr_real = pd.DataFrame(X_imputed).corr().fillna(0).values
corr_synth = df_synth.drop(columns=['patid']).corr().fillna(0).values
diff_corr = np.mean(np.abs(corr_real - corr_synth))
print(f"Différence des corrélations (Utilité)         : {diff_corr:.4f} (Viser le plus bas)")

#C. Confidentialité
indices_echantillon = np.random.choice(len(X_scaled), min(500, len(X_scaled)), replace=False)
dist_matrice = cdist(X_synth_scaled[indices_echantillon], X_scaled[indices_echantillon], metric='euclidean')
distance_knn_moyenne = np.mean(np.min(dist_matrice, axis=1))
print(f"Distance moyenne au 1-NN (Confidentialité)    : {distance_knn_moyenne:.4f} (Viser le plus haut)")

ÉVALUATION DES PERFORMANCES
Distance de Wasserstein moyenne (Utilité)       : 0.0980 (Viser le plus bas)
Différence des corrélations (Utilité)         : 0.0105 (Viser le plus bas)
Distance moyenne au 1-NN (Confidentialité)    : 0.8357 (Viser le plus haut)


### Résultats

Avec un paramètre `K_VOISINS = 5`, les métriques suivantes ont été obtenues :

*   **Distance de Wasserstein moyenne (Utilité)**: 0.0980 (L'objectif est de viser le plus bas, une valeur de 0.0980 indique une bonne préservation des distributions univariées.)

*   **Différence des corrélations (Utilité)**: 0.0105 (L'objectif est de viser le plus bas, une valeur très faible de 0.0105 suggère une excellente préservation des relations multivariées.)

*   **Distance moyenne au 1-NN (Confidentialité)**: 0.8357 (L'objectif est de viser le plus haut, une valeur de 0.8357 indique que les avatars sont suffisamment distincts des patients réels, contribuant à la confidentialité.)

Ces résultats montrent un bon équilibre. Les faibles valeurs pour les métriques d'utilité (Wasserstein et corrélations) indiquent que les données synthétiques conservent efficacement les caractéristiques statistiques et les relations des données réelles. La distance au plus proche voisin pour la confidentialité est également à un niveau raisonnable, suggérant que les avatars ne sont pas de simples copies des patients originaux.

### Conclusion

La méthode "Avatar" appliquée avec `K_VOISINS = 5` a permis de générer un jeu de données synthétiques (`syntheticDataDC2.csv`) qui démontre un excellent compromis entre utilité et confidentialité. Les distributions univariées et les corrélations multivariées sont bien préservées, tandis que la distance moyenne au plus proche voisin assure une protection adéquate de la vie privée des individus originaux.

Ce jeu de données synthétiques peut être utilisé pour diverses analyses et développements, réduisant ainsi les risques liés à l'utilisation directe de données sensibles. Des ajustements du paramètre `K_VOISINS` pourraient être explorés pour optimiser davantage ce compromis en fonction des exigences spécifiques du cas d'usage.

# 5. SAUVEGARDE DU JEU DE DONNÉES SYNTHÉTIQUES


In [ ]:
nom_fichier = "syntheticDataDC2.csv"
df_synth.to_csv(nom_fichier, index=False)
print(f"\nTerminé. Le fichier de données synthétiques a été sauvegardé sous : {nom_fichier}")


Terminé. Le fichier de données synthétiques a été sauvegardé sous : syntheticDataDC2.csv
